# Business Understanding

## FedEx Fuel Consumption and Price Forecasting

### Business Understanding

#### Business Problem

Fuel is a major operating cost for transportation and logistics companies such as FedEx. Changes in fuel consumption and fuel prices can therefore affect operational costs and overall business performance.

The objective of this project is to use historical fuel and related data to predict future fuel requirements and fuel prices.

#### Main Questions

1. Can we predict FedEx's fuel consumption for the following week using historical fuel consumption data?
2. Can we predict the following week's fuel price using historical fuel prices and crude oil prices?
3. How strongly is FedEx fuel cost related to crude oil and other fuel prices?
4. Which historical and external variables are most useful for predicting future fuel consumption and price?

#### Project Objectives

The project will:

- Analyze historical fuel consumption and fuel price trends.
- Investigate relationships between crude oil, fuel prices and FedEx fuel consumption.
- Create time-based and lagged features from historical data.
- Establish simple forecasting baselines.
- Train and compare machine learning/time-series models.
- Evaluate the models using appropriate forecasting metrics.
- Forecast the next week's fuel consumption and fuel price.
- Interpret the factors contributing to the forecasts.

#### Target Variables

The primary targets will be:

- Future fuel consumption
- Future fuel price

The exact target columns will be determined after inspecting the dataset.

#### Success Criteria

A successful model should perform better than a simple baseline forecast and provide useful predictions without using information that would not have been available at the time of prediction.

## 2. Importing Libraries

The following libraries will be used for data manipulation, visualization, statistical analysis and machine learning.

We will initially import only the libraries needed for data understanding and exploration. Additional libraries will be added later when required.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully.


## 3. Loading the Datasets

This project uses multiple datasets from different sources. Each dataset provides information that will contribute to understanding and forecasting fuel consumption and fuel prices.

The main datasets are:

1. **WTI Crude Oil Prices**
   - Provides daily West Texas Intermediate (WTI) crude oil prices.
   - Used as a macroeconomic indicator of movements in the oil market.

2. **FedEx International Fuel Surcharge**
   - Provides historical FedEx international fuel surcharge rates and the corresponding jet fuel price ranges.
   - Used to investigate the relationship between fuel prices and FedEx pricing.

3. **EIA State Fuel Prices**
   - Provides weekly retail gasoline and diesel prices by U.S. state.
   - Used to represent ground transportation fuel costs.

4. **California Household Travel Survey**
   - Provides household, vehicle and travel information.
   - May be used as a proxy for vehicle usage, travel distance and fuel-consumption patterns.

At this stage, the datasets will be loaded independently. No cleaning, transformation or merging will be performed until the structure and quality of each dataset have been understood.

In [5]:
DCOILWTICO_df = "../data/raw/DCOILWTICO.csv"
international_fuel_surcharge_df = "../data/raw/international-fuel-surcharge.csv"
state_prices_df = "../data/raw/state-prices.csv"

wti_df = pd.read_csv(DCOILWTICO_df)
fuel_surcharge_df = pd.read_csv(international_fuel_surcharge_df)
state_prices_df = pd.read_csv(state_prices_df)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [6]:
print("WTI Crude Oil Prices:")
display(wti_df.head())

print("\nFedEx International Fuel Surcharge:")
display(fuel_surcharge_df.head())

print("\nEIA State Fuel Prices:")
display(state_prices_df.head())

WTI Crude Oil Prices:


,observation_date,DCOILWTICO
0,2021-09-09,68.26
1,2021-09-10,69.82
2,2021-09-13,70.54
3,2021-09-14,70.53
4,2021-09-15,72.59



FedEx International Fuel Surcharge:


,Week,Week Price,Surcharge
0,"14 September, 2026 - 20 September, 2026",$4.08,49.00%
1,"07 September, 2026 - 13 September, 2026",$3.72,46.00%
2,"31 August, 2026 - 06 September, 2026",$3.92,47.75%
3,"24 August, 2026 - 30 August, 2026",$3.77,46.50%
4,"17 August, 2026 - 23 August, 2026",$3.43,43.50%



EIA State Fuel Prices:


,state,abbr,slug,padd,regular_price,regular_basis,regular_eia_area,regular_series,diesel_price,diesel_basis,diesel_eia_area,diesel_series,survey_week
0,Connecticut,CT,connecticut,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
1,Maine,ME,maine,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
2,Massachusetts,MA,massachusetts,1,4.068,state,Massachusetts,EMM_EPMR_PTE_SMA_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
3,New Hampshire,NH,new-hampshire,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
4,Rhode Island,RI,rhode-island,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31


### 3.1 Initial Dataset Structure

The datasets originate from different sources and therefore may have different structures, column names, date formats and levels of aggregation.

We will inspect the shape and columns of each dataset before proceeding with data preparation.

This allows us to determine:

- The number of observations and variables.
- The variables available for analysis.
- The time period covered.
- The level of aggregation.
- Which variables may be useful for forecasting.

In [8]:
print("WTI dataset shape:", wti_df.shape)
print("WTI columns:")
print(wti_df.columns.tolist())

print("\nFedEx fuel surcharge dataset shape:", fuel_surcharge_df.shape)
print("FedEx fuel surcharge columns:")
print(fuel_surcharge_df.columns.tolist())

print("\nState fuel prices dataset shape:", state_prices_df.shape)
print("State fuel prices columns:")
print(state_prices_df.columns.tolist())

WTI dataset shape: (1305, 2)
WTI columns:
['observation_date', 'DCOILWTICO']

FedEx fuel surcharge dataset shape: (13, 3)
FedEx fuel surcharge columns:
['Week', 'Week Price', 'Surcharge']

State fuel prices dataset shape: (51, 13)
State fuel prices columns:
['state', 'abbr', 'slug', 'padd', 'regular_price', 'regular_basis', 'regular_eia_area', 'regular_series', 'diesel_price', 'diesel_basis', 'diesel_eia_area', 'diesel_series', 'survey_week']


### 3.2 Dataset Information

We will examine the data types and non-null counts for each dataset.

This is an initial data-understanding step. We will not modify the data yet.

In [9]:
print("WTI Dataset Information")
wti_df.info()

print("\n" + "="*60 + "\n")

print("FedEx Fuel Surcharge Dataset Information")
fuel_surcharge_df.info()

print("\n" + "="*60 + "\n")

print("State Fuel Prices Dataset Information")
state_prices_df.info()

WTI Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 1305 entries, 0 to 1304
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  1305 non-null   str    
 1   DCOILWTICO        1248 non-null   float64
dtypes: float64(1), str(1)
memory usage: 20.5 KB


FedEx Fuel Surcharge Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Week        13 non-null     str  
 1   Week Price  13 non-null     str  
 2   Surcharge   13 non-null     str  
dtypes: str(3)
memory usage: 444.0 bytes


State Fuel Prices Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   state             51 non-null     str    
 

### 3.3 Previewing the Datasets

The first few observations provide an initial view of how the variables are represented.

We will inspect both the beginning and end of each dataset to understand the chronological coverage and identify any obvious structural differences.

In [10]:
print("WTI - First 5 rows")
display(wti_df.head())

print("WTI - Last 5 rows")
display(wti_df.tail())

print("FedEx Fuel Surcharge - First 5 rows")
display(fuel_surcharge_df.head())

print("FedEx Fuel Surcharge - Last 5 rows")
display(fuel_surcharge_df.tail())

print("State Prices - First 5 rows")
display(state_prices_df.head())

print("State Prices - Last 5 rows")
display(state_prices_df.tail())

WTI - First 5 rows


,observation_date,DCOILWTICO
0,2021-09-09,68.26
1,2021-09-10,69.82
2,2021-09-13,70.54
3,2021-09-14,70.53
4,2021-09-15,72.59


WTI - Last 5 rows


,observation_date,DCOILWTICO
1300,2026-09-03,92.55
1301,2026-09-04,92.69
1302,2026-09-07,NaN
1303,2026-09-08,94.21
1304,2026-09-09,97.26


FedEx Fuel Surcharge - First 5 rows


...

,Week,Week Price,Surcharge
8,"20 July, 2026 - 26 July, 2026",$2.97,39.75%
9,"13 July, 2026 - 19 July, 2026",$2.82,38.50%
10,"06 July, 2026 - 12 July, 2026",$2.79,38.25%
11,"29 June, 2026 - 05 July, 2026",$2.84,38.50%
12,"22 June, 2026 - 28 June, 2026",$3.19,41.50%


State Prices - First 5 rows


,state,abbr,slug,padd,regular_price,regular_basis,regular_eia_area,regular_series,diesel_price,diesel_basis,diesel_eia_area,diesel_series,survey_week
0,Connecticut,CT,connecticut,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
1,Maine,ME,maine,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
2,Massachusetts,MA,massachusetts,1,4.068,state,Massachusetts,EMM_EPMR_PTE_SMA_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
3,New Hampshire,NH,new-hampshire,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31
4,Rhode Island,RI,rhode-island,1,4.096,region,New England (PADD 1A),EMM_EPMR_PTE_R1X_DPG,5.736,region,New England (PADD 1A),EMD_EPD2D_PTE_R1X_DPG,2026-08-31


State Prices - Last 5 rows


,state,abbr,slug,padd,regular_price,regular_basis,regular_eia_area,regular_series,diesel_price,diesel_basis,diesel_eia_area,diesel_series,survey_week
46,California,CA,california,5,5.520,state,California,EMM_EPMR_PTE_SCA_DPG,7.218,state,California,EMD_EPD2D_PTE_SCA_DPG,2026-08-31
47,Hawaii,HI,hawaii,5,4.823,region,West Coast less California,EMM_EPMR_PTE_R5XCA_DPG,5.872,region,West Coast less California,EMD_EPD2D_PTE_R5XCA_DPG,2026-08-31
48,Nevada,NV,nevada,5,4.823,region,West Coast less California,EMM_EPMR_PTE_R5XCA_DPG,5.872,region,West Coast less California,EMD_EPD2D_PTE_R5XCA_DPG,2026-08-31
49,Oregon,OR,oregon,5,4.823,region,West Coast less California,EMM_EPMR_PTE_R5XCA_DPG,5.872,region,West Coast less California,EMD_EPD2D_PTE_R5XCA_DPG,2026-08-31
50,Washington,WA,washington,5,5.259,state,Washington,EMM_EPMR_PTE_SWA_DPG,5.872,region,West Coast less California,EMD_EPD2D_PTE_R5XCA_DPG,2026-08-31
